# Objective 3 — Step 12: Frozen Cost-Aware Threshold Replication

Step 11 selected the **MinCost FN:FP = 2:1 threshold-selection rule** using German Credit only.

## Frozen upstream framework

- Group-aware Chi-Square Top-75%
- Balanced Logistic Regression
- Balanced Random Forest
- Balanced XGBoost
- Equal-probability Soft Voting
- Sigmoid probability calibration

## Frozen decision rule

For every outer-training partition, choose the threshold that minimizes:

\[
C = FP + 2FN
\]

using **cross-fitted sigmoid-calibrated inner OOF probabilities only**.

The numeric threshold itself is **not frozen globally** because probability distributions differ across datasets and folds. What is frozen is the threshold-selection rule.

## Independent replication datasets

- Australian Credit Approval
- Taiwan Credit Card Default

No Australian or Taiwan outer-test labels are used to select thresholds.

The 2:1 loss ratio is a **reference asymmetric-cost scenario**, not a claim about a bank's actual monetary costs.


In [1]:
%pip install pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: C:\Users\hp\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:

from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")

BASE_DIR = Path(r"D:\PHD\Research Paper writing\3rd Obj. paper")

STEP10_DIR = BASE_DIR / "results" / "frozen_sigmoid_calibration_replication"
STEP11_DIR = BASE_DIR / "results" / "threshold_german_development"
DATA_DIR = BASE_DIR / "data" / "processed"

OUT_DIR = BASE_DIR / "results" / "frozen_cost_aware_threshold_replication"
OUT_DIR.mkdir(parents=True, exist_ok=True)

STEP10_OOF_FILE = (
    STEP10_DIR / "frozen_sigmoid_inner_oof_probabilities.csv"
)
STEP10_OUTER_FILE = (
    STEP10_DIR / "frozen_sigmoid_replication_outer_predictions.csv"
)
STEP11_SUMMARY_FILE = (
    STEP11_DIR / "german_threshold_development_summary.csv"
)

DATASET_FILES = {
    "Australian Credit Approval":
        DATA_DIR / "australian_credit_approval_cleaned.csv",
    "Taiwan Credit Card Default":
        DATA_DIR / "taiwan_credit_card_default_cleaned.csv",
}

required = [
    STEP10_OOF_FILE,
    STEP10_OUTER_FILE,
    STEP11_SUMMARY_FILE,
    *DATASET_FILES.values(),
]

missing = [str(path) for path in required if not path.exists()]

if missing:
    raise FileNotFoundError(
        "Required previous-step files are missing:\n"
        + "\n".join(missing)
    )

REPEAT_SEEDS = [42, 142, 242, 342, 442]
OUTER_FOLDS = 5
CALIBRATION_CROSSFIT_FOLDS = 5

EPS = 1e-6

FROZEN_THRESHOLD_RULE = "MinCost_FN2_FP1"

print("Output folder:", OUT_DIR)


Output folder: D:\PHD\Research Paper writing\3rd Obj. paper\results\frozen_cost_aware_threshold_replication


## 1. Verify and lock the German-developed threshold rule

In [3]:

step11_summary = pd.read_csv(
    STEP11_SUMMARY_FILE
)

selected_german = step11_summary[
    step11_summary["threshold_strategy"]
    == FROZEN_THRESHOLD_RULE
].copy()

if len(selected_german) != 1:
    raise RuntimeError(
        "German MinCost FN2:FP1 rule could not "
        "be uniquely identified."
    )

frozen_rule = {
    "development_dataset": "German Credit",
    "threshold_rule": (
        "Minimize FP + 2*FN on cross-fitted "
        "sigmoid-calibrated inner OOF probabilities"
    ),
    "reference_loss_ratio": "FN:FP = 2:1",
    "interpretation": (
        "Reference asymmetric-loss scenario only; "
        "not actual bank-specific monetary cost."
    ),
    "numeric_threshold_frozen_globally": False,
    "external_rule_retuning": False,
    "german_development_result": {
        "threshold_mean": float(
            selected_german[
                "Threshold_Mean"
            ].iloc[0]
        ),
        "threshold_sd": float(
            selected_german[
                "Threshold_SD"
            ].iloc[0]
        ),
        "recall": float(
            selected_german[
                "Recall"
            ].iloc[0]
        ),
        "precision": float(
            selected_german[
                "Precision"
            ].iloc[0]
        ),
        "f1": float(
            selected_german[
                "F1"
            ].iloc[0]
        ),
        "balanced_accuracy": float(
            selected_german[
                "Balanced_Accuracy"
            ].iloc[0]
        ),
        "mcc": float(
            selected_german[
                "MCC"
            ].iloc[0]
        ),
        "cost_2_1_per100": float(
            selected_german[
                "Cost_2_1"
            ].iloc[0]
        ),
    },
}

with open(
    OUT_DIR / "frozen_cost_aware_threshold_rule.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        frozen_rule,
        file,
        indent=4,
    )

print(json.dumps(frozen_rule, indent=2))


{
  "development_dataset": "German Credit",
  "threshold_rule": "Minimize FP + 2*FN on cross-fitted sigmoid-calibrated inner OOF probabilities",
  "reference_loss_ratio": "FN:FP = 2:1",
  "interpretation": "Reference asymmetric-loss scenario only; not actual bank-specific monetary cost.",
  "numeric_threshold_frozen_globally": false,
  "external_rule_retuning": false,
  "german_development_result": {
    "threshold_mean": 0.3537563925644884,
    "threshold_sd": 0.0407438327025447,
    "recall": 0.6893603943474447,
    "precision": 0.553559099685055,
    "f1": 0.6088080507206388,
    "balanced_accuracy": 0.7238120883870833,
    "mcc": 0.4253175003609249,
    "cost_2_1_per100": 35.72
  }
}


## 2. Load Step-10 probabilities and cleaned replication datasets

In [4]:

inner_oof = pd.read_csv(
    STEP10_OOF_FILE
)

outer_predictions = pd.read_csv(
    STEP10_OUTER_FILE
)

datasets = {
    dataset_name: pd.read_csv(path)
    for dataset_name, path in DATASET_FILES.items()
}

print(
    "Step-10 inner OOF rows:",
    len(inner_oof),
)

print(
    "Step-10 outer prediction rows:",
    len(outer_predictions),
)

for dataset_name, df in datasets.items():
    print(
        dataset_name,
        "| records =", len(df),
        "| adverse rate =",
        round(
            df["adverse_target"].mean(),
            4,
        ),
    )

assert set(
    inner_oof["dataset"].unique()
) == set(
    datasets.keys()
)

assert set(
    outer_predictions["dataset"].unique()
) == set(
    datasets.keys()
)


Step-10 inner OOF rows: 613800
Step-10 outer prediction rows: 153450
Australian Credit Approval | records = 690 | adverse rate = 0.5551
Taiwan Credit Card Default | records = 30000 | adverse rate = 0.2212


## 3. Recreate the exact outer folds

In [5]:

all_splits = {}

for dataset_name, df in datasets.items():
    y = (
        df["adverse_target"]
        .astype(int)
        .copy()
    )

    groups = (
        df["profile_group_id"]
        .astype(str)
        .copy()
    )

    dummy_X = np.zeros(
        (len(df), 1)
    )

    split_dict = {}

    for repeat_no, seed in enumerate(
        REPEAT_SEEDS,
        start=1,
    ):
        splitter = StratifiedGroupKFold(
            n_splits=OUTER_FOLDS,
            shuffle=True,
            random_state=seed,
        )

        for fold_no, (
            train_idx,
            test_idx,
        ) in enumerate(
            splitter.split(
                dummy_X,
                y,
                groups,
            ),
            start=1,
        ):
            run_id = (
                f"R{repeat_no}_F{fold_no}"
            )

            train_groups = set(
                groups.iloc[
                    train_idx
                ]
            )

            test_groups = set(
                groups.iloc[
                    test_idx
                ]
            )

            assert len(
                train_groups.intersection(
                    test_groups
                )
            ) == 0

            split_dict[run_id] = {
                "repeat": repeat_no,
                "fold": fold_no,
                "seed": seed,
                "train_idx": np.asarray(
                    train_idx,
                    dtype=int,
                ),
                "test_idx": np.asarray(
                    test_idx,
                    dtype=int,
                ),
            }

    all_splits[
        dataset_name
    ] = split_dict


for dataset_name in datasets:
    print(
        dataset_name,
        "outer runs =",
        len(
            all_splits[
                dataset_name
            ]
        ),
    )


Australian Credit Approval outer runs = 25
Taiwan Credit Card Default outer runs = 25


## 4. Sigmoid helper functions

In [6]:

def clip_probability(probability):
    return np.clip(
        np.asarray(
            probability,
            dtype=float,
        ),
        EPS,
        1.0 - EPS,
    )


def logit(probability):
    probability = (
        clip_probability(
            probability
        )
    )

    return np.log(
        probability
        / (
            1.0
            - probability
        )
    )


def fit_sigmoid(
    raw_probability,
    y_true,
):
    model = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=5000,
        random_state=42,
    )

    model.fit(
        logit(
            raw_probability
        ).reshape(
            -1,
            1,
        ),
        np.asarray(
            y_true,
            dtype=int,
        ),
    )

    return model


def apply_sigmoid(
    model,
    raw_probability,
):
    return model.predict_proba(
        logit(
            raw_probability
        ).reshape(
            -1,
            1,
        )
    )[:, 1]


## 5. Cross-fit the sigmoid calibrator inside every outer-training partition

This reproduces the Step-11 threshold-development logic without using external test labels.


In [7]:

crossfit_rows = []

for dataset_name, df in datasets.items():
    y_full = (
        df["adverse_target"]
        .astype(int)
        .copy()
    )

    groups_full = (
        df["profile_group_id"]
        .astype(str)
        .copy()
    )

    for run_id, info in (
        all_splits[
            dataset_name
        ].items()
    ):
        train_idx = (
            info["train_idx"]
        )

        y_outer_train = (
            y_full.iloc[
                train_idx
            ]
            .reset_index(
                drop=True
            )
        )

        groups_outer_train = (
            groups_full.iloc[
                train_idx
            ]
            .reset_index(
                drop=True
            )
        )

        run_oof = (
            inner_oof[
                (
                    inner_oof[
                        "dataset"
                    ]
                    == dataset_name
                )
                & (
                    inner_oof[
                        "run_id"
                    ]
                    == run_id
                )
            ]
            .sort_values(
                "outer_train_position"
            )
            .reset_index(
                drop=True
            )
        )

        assert len(
            run_oof
        ) == len(
            train_idx
        )

        expected_positions = np.arange(
            len(
                train_idx
            )
        )

        assert np.array_equal(
            run_oof[
                "outer_train_position"
            ].to_numpy(
                dtype=int
            ),
            expected_positions,
        )

        assert np.array_equal(
            run_oof[
                "y_true"
            ]
            .astype(int)
            .to_numpy(),
            y_outer_train.to_numpy(),
        )

        raw_oof_probability = (
            run_oof[
                "raw_oof_hybrid_probability"
            ]
            .to_numpy(
                dtype=float
            )
        )

        calibrated_crossfit = (
            np.full(
                len(
                    run_oof
                ),
                np.nan,
                dtype=float,
            )
        )

        splitter = (
            StratifiedGroupKFold(
                n_splits=(
                    CALIBRATION_CROSSFIT_FOLDS
                ),
                shuffle=True,
                random_state=(
                    50000
                    + info["seed"]
                    + info["fold"]
                ),
            )
        )

        dummy = np.zeros(
            (
                len(
                    run_oof
                ),
                1,
            )
        )

        for calibration_fold, (
            calibration_train_pos,
            calibration_valid_pos,
        ) in enumerate(
            splitter.split(
                dummy,
                y_outer_train,
                groups_outer_train,
            ),
            start=1,
        ):
            calibrator = fit_sigmoid(
                raw_oof_probability[
                    calibration_train_pos
                ],
                y_outer_train.iloc[
                    calibration_train_pos
                ].to_numpy(),
            )

            calibrated_values = (
                apply_sigmoid(
                    calibrator,
                    raw_oof_probability[
                        calibration_valid_pos
                    ],
                )
            )

            calibrated_crossfit[
                calibration_valid_pos
            ] = calibrated_values

            for local_position, (
                outer_train_position
            ) in enumerate(
                calibration_valid_pos
            ):
                crossfit_rows.append({
                    "dataset": (
                        dataset_name
                    ),
                    "run_id": run_id,
                    "repeat": (
                        info[
                            "repeat"
                        ]
                    ),
                    "fold": (
                        info[
                            "fold"
                        ]
                    ),
                    "calibration_crossfit_fold": (
                        calibration_fold
                    ),
                    "outer_train_position": int(
                        outer_train_position
                    ),
                    "y_true": int(
                        y_outer_train.iloc[
                            outer_train_position
                        ]
                    ),
                    "raw_oof_probability": float(
                        raw_oof_probability[
                            outer_train_position
                        ]
                    ),
                    "crossfitted_sigmoid_probability": float(
                        calibrated_values[
                            local_position
                        ]
                    ),
                })

        assert np.isfinite(
            calibrated_crossfit
        ).all()


crossfit_oof = pd.DataFrame(
    crossfit_rows
)

crossfit_oof.to_csv(
    OUT_DIR
    / "frozen_threshold_crossfitted_calibrated_oof.csv",
    index=False,
)

print(
    "Cross-fitted OOF rows:",
    len(
        crossfit_oof
    ),
)


Cross-fitted OOF rows: 613800


## 6. Frozen FN:FP = 2:1 threshold-selection rule

In [8]:

def threshold_metrics(
    y_true,
    probability,
    threshold,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    probability = np.asarray(
        probability,
        dtype=float,
    )

    prediction = (
        probability
        >= threshold
    ).astype(int)

    tn, fp, fn, tp = (
        confusion_matrix(
            y_true,
            prediction,
            labels=[
                0,
                1,
            ],
        ).ravel()
    )

    specificity = (
        tn
        / (
            tn
            + fp
        )
        if (
            tn
            + fp
        ) > 0
        else np.nan
    )

    sensitivity = (
        tp
        / (
            tp
            + fn
        )
        if (
            tp
            + fn
        ) > 0
        else np.nan
    )

    gmean = (
        math.sqrt(
            specificity
            * sensitivity
        )
        if not np.isnan(
            specificity
            + sensitivity
        )
        else np.nan
    )

    return {
        "threshold": float(
            threshold
        ),
        "accuracy": accuracy_score(
            y_true,
            prediction,
        ),
        "precision_adverse": precision_score(
            y_true,
            prediction,
            pos_label=1,
            zero_division=0,
        ),
        "recall_adverse": recall_score(
            y_true,
            prediction,
            pos_label=1,
            zero_division=0,
        ),
        "specificity": specificity,
        "f1_adverse": f1_score(
            y_true,
            prediction,
            pos_label=1,
            zero_division=0,
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true,
                prediction,
            )
        ),
        "mcc": matthews_corrcoef(
            y_true,
            prediction,
        ),
        "gmean": gmean,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "cost_FN1_FP1": int(
            fp
            + fn
        ),
        "cost_FN2_FP1": int(
            fp
            + 2
            * fn
        ),
        "cost_FN5_FP1": int(
            fp
            + 5
            * fn
        ),
        "cost_FN10_FP1": int(
            fp
            + 10
            * fn
        ),
    }


def candidate_thresholds(
    probability,
):
    values = np.unique(
        clip_probability(
            probability
        )
    )

    if len(
        values
    ) == 1:
        return np.array(
            [
                0.5
            ]
        )

    midpoints = (
        values[:-1]
        + values[1:]
    ) / 2.0

    return np.unique(
        np.concatenate([
            np.array([
                EPS,
                0.5,
                1.0 - EPS,
            ]),
            midpoints,
        ])
    )


def choose_min_cost_2_to_1_threshold(
    y_true,
    probability,
):
    rows = [
        threshold_metrics(
            y_true,
            probability,
            threshold,
        )
        for threshold
        in candidate_thresholds(
            probability
        )
    ]

    table = pd.DataFrame(
        rows
    )

    ranked = table.sort_values(
        [
            "cost_FN2_FP1",
            "mcc",
            "balanced_accuracy",
            "f1_adverse",
        ],
        ascending=[
            True,
            False,
            False,
            False,
        ],
    )

    return float(
        ranked.iloc[
            0
        ][
            "threshold"
        ]
    )


## 7. Select thresholds from training OOF and evaluate external outer-test probabilities

In [9]:

threshold_rows = []
result_rows = []
prediction_rows = []

for dataset_name, df in datasets.items():
    for run_id, info in (
        all_splits[
            dataset_name
        ].items()
    ):
        run_crossfit = (
            crossfit_oof[
                (
                    crossfit_oof[
                        "dataset"
                    ]
                    == dataset_name
                )
                & (
                    crossfit_oof[
                        "run_id"
                    ]
                    == run_id
                )
            ]
            .sort_values(
                "outer_train_position"
            )
        )

        y_train_oof = (
            run_crossfit[
                "y_true"
            ]
            .astype(int)
            .to_numpy()
        )

        calibrated_train_oof = (
            run_crossfit[
                "crossfitted_sigmoid_probability"
            ]
            .to_numpy(
                dtype=float
            )
        )

        selected_threshold = (
            choose_min_cost_2_to_1_threshold(
                y_train_oof,
                calibrated_train_oof,
            )
        )

        training_metrics = (
            threshold_metrics(
                y_train_oof,
                calibrated_train_oof,
                selected_threshold,
            )
        )

        run_outer = (
            outer_predictions[
                (
                    outer_predictions[
                        "dataset"
                    ]
                    == dataset_name
                )
                & (
                    outer_predictions[
                        "run_id"
                    ]
                    == run_id
                )
            ]
            .copy()
        )

        y_outer_test = (
            run_outer[
                "y_true"
            ]
            .astype(int)
            .to_numpy()
        )

        calibrated_outer_test = (
            run_outer[
                "sigmoid_probability"
            ]
            .to_numpy(
                dtype=float
            )
        )

        threshold_rows.append({
            "dataset": dataset_name,
            "run_id": run_id,
            "repeat": (
                info[
                    "repeat"
                ]
            ),
            "fold": (
                info[
                    "fold"
                ]
            ),
            "threshold_rule": (
                FROZEN_THRESHOLD_RULE
            ),
            "selected_threshold": (
                selected_threshold
            ),
            "training_oof_recall": (
                training_metrics[
                    "recall_adverse"
                ]
            ),
            "training_oof_precision": (
                training_metrics[
                    "precision_adverse"
                ]
            ),
            "training_oof_f1": (
                training_metrics[
                    "f1_adverse"
                ]
            ),
            "training_oof_balanced_accuracy": (
                training_metrics[
                    "balanced_accuracy"
                ]
            ),
            "training_oof_mcc": (
                training_metrics[
                    "mcc"
                ]
            ),
            "training_oof_cost_FN2_FP1": (
                training_metrics[
                    "cost_FN2_FP1"
                ]
            ),
        })

        for strategy, threshold in {
            "Fixed0.50": 0.50,
            FROZEN_THRESHOLD_RULE: (
                selected_threshold
            ),
        }.items():
            metrics = (
                threshold_metrics(
                    y_outer_test,
                    calibrated_outer_test,
                    threshold,
                )
            )

            n_test = len(
                y_outer_test
            )

            result_rows.append({
                "dataset": (
                    dataset_name
                ),
                "run_id": run_id,
                "repeat": (
                    info[
                        "repeat"
                    ]
                ),
                "fold": (
                    info[
                        "fold"
                    ]
                ),
                "threshold_strategy": (
                    strategy
                ),
                "selected_threshold": (
                    threshold
                ),
                **metrics,
                "cost_FN1_FP1_per100": (
                    100.0
                    * metrics[
                        "cost_FN1_FP1"
                    ]
                    / n_test
                ),
                "cost_FN2_FP1_per100": (
                    100.0
                    * metrics[
                        "cost_FN2_FP1"
                    ]
                    / n_test
                ),
                "cost_FN5_FP1_per100": (
                    100.0
                    * metrics[
                        "cost_FN5_FP1"
                    ]
                    / n_test
                ),
                "cost_FN10_FP1_per100": (
                    100.0
                    * metrics[
                        "cost_FN10_FP1"
                    ]
                    / n_test
                ),
            })

        final_prediction = (
            calibrated_outer_test
            >= selected_threshold
        ).astype(int)

        fixed_prediction = (
            calibrated_outer_test
            >= 0.50
        ).astype(int)

        for local_position, (
            source_row_index
        ) in enumerate(
            run_outer[
                "source_row_index"
            ].to_numpy(
                dtype=int
            )
        ):
            prediction_rows.append({
                "dataset": (
                    dataset_name
                ),
                "run_id": run_id,
                "repeat": (
                    info[
                        "repeat"
                    ]
                ),
                "fold": (
                    info[
                        "fold"
                    ]
                ),
                "source_row_index": int(
                    source_row_index
                ),
                "y_true": int(
                    y_outer_test[
                        local_position
                    ]
                ),
                "sigmoid_probability": float(
                    calibrated_outer_test[
                        local_position
                    ]
                ),
                "fixed_0_5_prediction": int(
                    fixed_prediction[
                        local_position
                    ]
                ),
                "frozen_cost_aware_threshold": float(
                    selected_threshold
                ),
                "frozen_cost_aware_prediction": int(
                    final_prediction[
                        local_position
                    ]
                ),
            })


thresholds = pd.DataFrame(
    threshold_rows
)

fold_results = pd.DataFrame(
    result_rows
)

predictions = pd.DataFrame(
    prediction_rows
)

thresholds.to_csv(
    OUT_DIR
    / "frozen_cost_aware_thresholds_by_run.csv",
    index=False,
)

fold_results.to_csv(
    OUT_DIR
    / "frozen_cost_aware_threshold_fold_results.csv",
    index=False,
)

predictions.to_csv(
    OUT_DIR
    / "frozen_cost_aware_outer_predictions.csv",
    index=False,
)

print(
    "Threshold selections:",
    len(
        thresholds
    ),
)

print(
    "Fold-result rows:",
    len(
        fold_results
    ),
)


Threshold selections: 50
Fold-result rows: 100


## 8. Independent replication summary

In [10]:

summary = (
    fold_results
    .groupby(
        [
            "dataset",
            "threshold_strategy",
        ],
        as_index=False,
    )
    .agg(
        Threshold_Mean=(
            "selected_threshold",
            "mean",
        ),
        Threshold_SD=(
            "selected_threshold",
            "std",
        ),
        Threshold_Min=(
            "selected_threshold",
            "min",
        ),
        Threshold_Max=(
            "selected_threshold",
            "max",
        ),
        Recall=(
            "recall_adverse",
            "mean",
        ),
        Precision=(
            "precision_adverse",
            "mean",
        ),
        F1=(
            "f1_adverse",
            "mean",
        ),
        Specificity=(
            "specificity",
            "mean",
        ),
        Balanced_Accuracy=(
            "balanced_accuracy",
            "mean",
        ),
        MCC=(
            "mcc",
            "mean",
        ),
        GMean=(
            "gmean",
            "mean",
        ),
        Cost_1_1=(
            "cost_FN1_FP1_per100",
            "mean",
        ),
        Cost_2_1=(
            "cost_FN2_FP1_per100",
            "mean",
        ),
        Cost_5_1=(
            "cost_FN5_FP1_per100",
            "mean",
        ),
        Cost_10_1=(
            "cost_FN10_FP1_per100",
            "mean",
        ),
    )
)

summary.to_csv(
    OUT_DIR
    / "frozen_cost_aware_threshold_replication_summary.csv",
    index=False,
)

display(summary)


,dataset,threshold_strategy,Threshold_Mean,Threshold_SD,Threshold_Min,Threshold_Max,Recall,Precision,F1,Specificity,Balanced_Accuracy,MCC,GMean,Cost_1_1,Cost_2_1,Cost_5_1,Cost_10_1
0,Australian Credit Approval,Fixed0.50,0.500000,0.000000,0.500000,0.500000,0.864474,0.892598,0.877628,0.871041,0.867758,0.732658,0.867344,13.304348,20.840580,43.449275,81.130435
1,Australian Credit Approval,MinCost_FN2_FP1,0.328367,0.058577,0.224689,0.432040,0.918994,0.832245,0.872189,0.768729,0.843862,0.702414,0.839363,14.869565,19.391304,32.956522,55.565217
2,Taiwan Credit Card Default,Fixed0.50,0.500000,0.000000,0.500000,0.500000,0.363076,0.676515,0.472411,0.950635,0.656856,0.402540,0.587425,17.934665,32.024664,74.294659,144.744650
3,Taiwan Credit Card Default,MinCost_FN2_FP1,0.316157,0.013658,0.289545,0.341883,0.511119,0.570413,0.538789,0.890444,0.700782,0.418124,0.674477,19.347967,30.163330,62.609417,116.686229


## 9. Paired deltas against calibrated threshold 0.50

In [11]:

reference = (
    fold_results[
        fold_results[
            "threshold_strategy"
        ]
        == "Fixed0.50"
    ]
    [
        [
            "dataset",
            "run_id",
            "recall_adverse",
            "precision_adverse",
            "f1_adverse",
            "balanced_accuracy",
            "mcc",
            "gmean",
            "cost_FN1_FP1_per100",
            "cost_FN2_FP1_per100",
            "cost_FN5_FP1_per100",
            "cost_FN10_FP1_per100",
        ]
    ]
    .copy()
)

reference = reference.rename(
    columns={
        column: (
            "reference_"
            + column
        )
        for column
        in reference.columns
        if column
        not in [
            "dataset",
            "run_id",
        ]
    }
)

paired = fold_results.merge(
    reference,
    on=[
        "dataset",
        "run_id",
    ],
    how="left",
    validate="many_to_one",
)

for metric in [
    "recall_adverse",
    "precision_adverse",
    "f1_adverse",
    "balanced_accuracy",
    "mcc",
    "gmean",
    "cost_FN1_FP1_per100",
    "cost_FN2_FP1_per100",
    "cost_FN5_FP1_per100",
    "cost_FN10_FP1_per100",
]:
    paired[
        "delta_"
        + metric
    ] = (
        paired[
            metric
        ]
        - paired[
            "reference_"
            + metric
        ]
    )

paired.to_csv(
    OUT_DIR
    / "frozen_cost_aware_threshold_paired_deltas.csv",
    index=False,
)

delta_summary = (
    paired[
        paired[
            "threshold_strategy"
        ]
        == FROZEN_THRESHOLD_RULE
    ]
    .groupby(
        "dataset",
        as_index=False,
    )
    .agg(
        Delta_Recall=(
            "delta_recall_adverse",
            "mean",
        ),
        Delta_Precision=(
            "delta_precision_adverse",
            "mean",
        ),
        Delta_F1=(
            "delta_f1_adverse",
            "mean",
        ),
        Delta_Balanced_Accuracy=(
            "delta_balanced_accuracy",
            "mean",
        ),
        Delta_MCC=(
            "delta_mcc",
            "mean",
        ),
        Delta_GMean=(
            "delta_gmean",
            "mean",
        ),
        Delta_Cost_1_1=(
            "delta_cost_FN1_FP1_per100",
            "mean",
        ),
        Delta_Cost_2_1=(
            "delta_cost_FN2_FP1_per100",
            "mean",
        ),
        Delta_Cost_5_1=(
            "delta_cost_FN5_FP1_per100",
            "mean",
        ),
        Delta_Cost_10_1=(
            "delta_cost_FN10_FP1_per100",
            "mean",
        ),
    )
)

delta_summary.to_csv(
    OUT_DIR
    / "frozen_cost_aware_threshold_delta_summary.csv",
    index=False,
)

display(
    delta_summary
)


,dataset,Delta_Recall,Delta_Precision,Delta_F1,Delta_Balanced_Accuracy,Delta_MCC,Delta_GMean,Delta_Cost_1_1,Delta_Cost_2_1,Delta_Cost_5_1,Delta_Cost_10_1
0,Australian Credit Approval,0.054519,-0.060353,-0.005440,-0.023896,-0.030245,-0.027981,1.565217,-1.449275,-10.492754,-25.565217
1,Taiwan Credit Card Default,0.148043,-0.106102,0.066377,0.043926,0.015584,0.087052,1.413302,-1.861334,-11.685242,-28.058421


## 10. Combined German + external operating-point table

In [12]:

german_reference = (
    step11_summary[
        step11_summary[
            "threshold_strategy"
        ]
        .isin([
            "Fixed0.50",
            FROZEN_THRESHOLD_RULE,
        ])
    ]
    .copy()
)

german_reference.insert(
    0,
    "dataset",
    "German Credit",
)

german_reference = german_reference.rename(
    columns={
        "threshold_strategy": (
            "threshold_strategy"
        ),
    }
)

external_summary = (
    summary.copy()
)

common_columns = [
    "dataset",
    "threshold_strategy",
    "Threshold_Mean",
    "Threshold_SD",
    "Threshold_Min",
    "Threshold_Max",
    "Recall",
    "Precision",
    "F1",
    "Specificity",
    "Balanced_Accuracy",
    "MCC",
    "GMean",
    "Cost_1_1",
    "Cost_2_1",
    "Cost_5_1",
    "Cost_10_1",
]

combined = pd.concat(
    [
        german_reference[
            common_columns
        ],
        external_summary[
            common_columns
        ],
    ],
    ignore_index=True,
)

combined.to_csv(
    OUT_DIR
    / "combined_cross_dataset_threshold_summary.csv",
    index=False,
)

display(
    combined
)


,dataset,threshold_strategy,Threshold_Mean,Threshold_SD,Threshold_Min,Threshold_Max,Recall,Precision,F1,Specificity,Balanced_Accuracy,MCC,GMean,Cost_1_1,Cost_2_1,Cost_5_1,Cost_10_1
0,German Credit,Fixed0.50,0.500000,0.000000,0.500000,0.500000,0.464135,0.646891,0.536208,0.891365,0.677750,0.395607,0.640875,23.740000,39.860000,88.220000,168.820000
1,German Credit,MinCost_FN2_FP1,0.353756,0.040744,0.233351,0.414247,0.689360,0.553559,0.608808,0.758264,0.723812,0.425318,0.720324,26.340000,35.720000,63.860000,110.760000
2,Australian Credit Approval,Fixed0.50,0.500000,0.000000,0.500000,0.500000,0.864474,0.892598,0.877628,0.871041,0.867758,0.732658,0.867344,13.304348,20.840580,43.449275,81.130435
3,Australian Credit Approval,MinCost_FN2_FP1,0.328367,0.058577,0.224689,0.432040,0.918994,0.832245,0.872189,0.768729,0.843862,0.702414,0.839363,14.869565,19.391304,32.956522,55.565217
4,Taiwan Credit Card Default,Fixed0.50,0.500000,0.000000,0.500000,0.500000,0.363076,0.676515,0.472411,0.950635,0.656856,0.402540,0.587425,17.934665,32.024664,74.294659,144.744650
5,Taiwan Credit Card Default,MinCost_FN2_FP1,0.316157,0.013658,0.289545,0.341883,0.511119,0.570413,0.538789,0.890444,0.700782,0.418124,0.674477,19.347967,30.163330,62.609417,116.686229


## 11. Final checks

Do not change the FN:FP = 2:1 rule based on Australian or Taiwan results.

The next stage after review will consolidate the complete proposed framework and perform:
- final paired statistical comparisons,
- ablation synthesis,
- explainability,
- Taiwan fairness audit,
- and manuscript-ready tables/figures.


In [13]:

assert len(
    thresholds
) == (
    2
    * 25
)

assert len(
    fold_results
) == (
    2
    * 25
    * 2
)

assert thresholds[
    "selected_threshold"
].between(
    0.0,
    1.0,
).all()

assert fold_results[
    "mcc"
].between(
    -1.0,
    1.0,
).all()

for metric in [
    "recall_adverse",
    "precision_adverse",
    "f1_adverse",
    "balanced_accuracy",
    "gmean",
]:
    assert fold_results[
        metric
    ].between(
        0.0,
        1.0,
    ).all()

configuration = {
    "stage": (
        "Objective 3 Step 12 - "
        "frozen cost-aware threshold replication"
    ),
    "development_dataset": (
        "German Credit"
    ),
    "replication_datasets": (
        list(
            datasets.keys()
        )
    ),
    "frozen_threshold_rule": (
        "Minimize FP + 2*FN"
    ),
    "threshold_training_data": (
        "Cross-fitted sigmoid-calibrated inner OOF "
        "hybrid probabilities from each outer-training fold"
    ),
    "threshold_numeric_value": (
        "Adapted from each training partition; not globally fixed"
    ),
    "outer_test_used_for_threshold_selection": False,
    "cost_interpretation": (
        "Reference asymmetric-loss scenario, "
        "not actual monetary bank cost"
    ),
}

with open(
    OUT_DIR
    / "step12_experiment_configuration.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        configuration,
        file,
        indent=4,
    )

manifest = sorted(
    [
        path.name
        for path
        in OUT_DIR.iterdir()
        if path.is_file()
    ]
)

pd.DataFrame(
    {
        "generated_file": manifest
    }
).to_csv(
    OUT_DIR
    / "step12_output_manifest.csv",
    index=False,
)

print("=" * 80)
print("STEP 12 COMPLETED SUCCESSFULLY")
print("=" * 80)
print("Output folder:", OUT_DIR)
print("\nMost important files:")
print(" - frozen_cost_aware_threshold_replication_summary.csv")
print(" - frozen_cost_aware_threshold_delta_summary.csv")
print(" - combined_cross_dataset_threshold_summary.csv")
print(" - frozen_cost_aware_thresholds_by_run.csv")
print(" - frozen_cost_aware_outer_predictions.csv")


STEP 12 COMPLETED SUCCESSFULLY
Output folder: D:\PHD\Research Paper writing\3rd Obj. paper\results\frozen_cost_aware_threshold_replication

Most important files:
 - frozen_cost_aware_threshold_replication_summary.csv
 - frozen_cost_aware_threshold_delta_summary.csv
 - combined_cross_dataset_threshold_summary.csv
 - frozen_cost_aware_thresholds_by_run.csv
 - frozen_cost_aware_outer_predictions.csv
